# Top-down walkthrough of the Simple Steps object model

This notebook builds the whole hierarchy **from the top down** and shows `info()` /
`repr()` at every layer, so you can *see* and *test* each object as you go.

```
App -> AppConfig, Tools, Resources, Sessions
  Session -> Workflows
    Workflow -> Steps (columns)
      Step = Operation (Tool + how) + Data (status/output)
```

Mirrors [`top_down_walkthrough.py`](top_down_walkthrough.py) and
[`docs/object-model.md`](../docs/object-model.md).

Every object exposes:

- `repr(obj)` — one dense line, safe for logs (no payloads).
- `obj.info()` — a rich `SummaryTable` (HTML here, aligned text in logs).

## 0. Developers define Tools

A **Tool** is a plain Python function registered with `@register_tool`. The
signature *is* the schema — parameters and type hints are introspected for you.

In [ ]:
from simple_steps_core import (
    App,
    AppConfig,
    Operation,
    OrchestrationConfig,
    Resource,
    StageExecutionConfig,
    StepExecutionConfig,
    WorkflowExecutionConfig,
    register_tool,
)

In [ ]:
@register_tool("make_list", description="Create the list [0, 1, ..., n-1].")
def make_list(n: int) -> list[int]:
    return list(range(n))


@register_tool("scale", description="Multiply a number by a factor.")
def scale(x: int, factor: int = 2) -> int:
    return x * factor


@register_tool("summarize", description="Count and total a list of numbers.")
def summarize(rows: list[int]) -> dict:
    return {"count": len(rows), "total": sum(rows)}


@register_tool("load_rows", description="Load n rows from the injected 'db' resource.")
def load_rows(n: int, db=Resource()) -> list[int]:
    return db.fetch(n)

`load_rows` declares `db=Resource()` — an **injected dependency**. The runtime
supplies it at call time from the session's resource registry, so the tool body
never has to know how the connection is built.

Here is the stand-in "database" that will back it:

In [ ]:
class FakeDB:
    """Stand-in external resource (a 'database')."""

    def fetch(self, n: int) -> list[int]:
        return list(range(100, 100 + n))

    def close(self) -> None:
        pass


def make_db() -> FakeDB:
    return FakeDB()

## 1. App + AppConfig — the whole system

The `App` is the top of the hierarchy: it owns the config, the tool registry,
default resources, and the live sessions.

In [ ]:
app = App(AppConfig(title="Tutorial App", port=8123))
app

In [ ]:
app.info()

## 2. Tools — the registered capabilities

`tools_info()` lists everything in the registry; a single definition's `info()`
shows the introspected parameter schema.

In [ ]:
app.tools_info()

In [ ]:
app.registry.get_definition("scale").info()

## 3. Session — one user's isolated area

A `Session` scopes workflows and resources to a single user. Sessions do not
share state with each other.

In [ ]:
session = app.session("alice")
session

## 4. Resources — declare → load → check → use

Resources are **lazy**: `register()` only declares how to build one. The optional
`check` is a cheap hello-world you can run before a long batch to fail fast.

In [ ]:
session.resources.register("db", make_db, check=lambda db: db.fetch(1))
session.resources.info()   # status: declared (lazy — nothing built yet)

In [ ]:
session.resources.check_all()   # builds each resource and runs its check

In [ ]:
session.resources.info()   # now loaded

## 5. Workflow — author Operations (Tool + how)

An **Operation** pairs a tool with *how* to call it: its arguments, which stage
it belongs to, and any orchestration. `step3` below uses `mode="map"` to fan the
`scale` tool out over every item produced by `step1`.

Argument references like `"step3.ok"` wire steps together — `step4` consumes the
successful outputs of `step3`.

In [ ]:
wf = session.workflow("demo")
wf.add(Operation(step_id="step1", name="make_list", stage="load", arguments={"n": 5}))
wf.add(Operation(step_id="step2", name="load_rows", stage="load", arguments={"n": 3}))
wf.add(Operation(
    step_id="step3", name="scale", stage="transform",
    arguments={"factor": 10},
    orchestration=OrchestrationConfig(mode="map", over="step1", concurrency=4),
))
wf.add(Operation(step_id="step4", name="summarize", stage="report",
                 arguments={"rows": "step3.ok"}))

# Isolated, per-scope execution configs (no inheritance / overriding).
wf.stage_config["transform"] = StageExecutionConfig(steps="parallel", concurrency=4)
wf.execution = WorkflowExecutionConfig(on_stage_error="stop")
wf

In [ ]:
wf.validate()   # dry-run preflight: tools exist, args typecheck, resources present

In [ ]:
wf.info()   # the spreadsheet view, BEFORE running

### 5b. Preflight catches a missing resource

`validate()` is worth running first. A fresh session has no `db` registered, so
the preflight reports the problem instead of failing halfway through a run.

In [ ]:
bob = app.session("bob")          # fresh session, no 'db'
wf_bob = bob.workflow("demo")
wf_bob.add(Operation(step_id="step1", name="load_rows", arguments={"n": 2}))
wf_bob.validate()

## 6. Run + inspect

Running fills in each `Step`'s **Data** side: status, output, and timing.

> **In a notebook, use `await wf.arun()`.** Jupyter already has a running event
> loop, and the synchronous `wf.run()` refuses to nest inside one whenever a step
> is async or orchestrated (`step3` is a `map`). In a plain script — where no loop
> is running — `wf.run()` is the right call. Same for `run_step` / `arun_step`
> and `run_by_stages` / `arun_by_stages`.

In [ ]:
await wf.arun()
wf.info()   # completed, with durations

In [ ]:
wf["step3"]   # Step repr: status + tool + duration

In [ ]:
wf.preview("step4")   # payload-free output preview

### 6b. Orchestrator output (MapResult)

A mapped step returns a `MapResult`, which is **partial-failure aware**: `ok`
holds the successes and `failed` the errors, so one bad item does not discard
the rest of the batch.

In [ ]:
result = wf["step3"].output.value
print(repr(result))
print("ok     :", result.ok)
print("failed :", result.failed)
print("summary:", wf["step4"].output.value)

## 7. Stages as views

A **Stage** is not a container — it is a *view* over the workflow's steps, like a
groupby. The steps are the single source of truth.

In [ ]:
for st in wf.stage_views():
    print(" ", repr(st))

In [ ]:
wf.stage("transform").info()

## 8. Session & App overview after work

The same `info()` calls from the top now roll up the work that was done.

In [ ]:
session.info()

In [ ]:
app.info()

## 9. Execution configs are orthogonal & isolated

Each scope has its own config object. There is **no inheritance or overriding** —
a step config never silently reaches up to a stage or workflow config.

In [ ]:
print("step    :", StepExecutionConfig(timeout=30, retries=2, cache=True))
print("stage   :", StageExecutionConfig(steps="parallel", concurrency=8))
print("workflow:", WorkflowExecutionConfig(on_stage_error="continue"))